# 01 - EDA

Explores the OULAD raw tables for one presentation and checks cohort size, outcomes, VLE activity types, and collaboration activity volume.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src import ingest
from src.config import raw_data_dir

PRESENTATION = ("AAA", "2014J")
DATA_DIR = raw_data_dir()
DATA_DIR

In [ ]:
tables = ingest.run(*PRESENTATION)
pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": len(df.columns)} for name, df in tables.items()]
).sort_values("table")

In [ ]:
tables["info"].head()

In [ ]:
outcomes = tables["info"]["final_result"].value_counts().rename_axis("final_result").reset_index(name="n")
outcomes["pct"] = outcomes["n"] / outcomes["n"].sum()
outcomes

In [ ]:
ax = outcomes.plot.bar(x="final_result", y="n", legend=False, title="Outcome Distribution")
ax.set_xlabel("Final result")
ax.set_ylabel("Learners")
plt.tight_layout()

In [ ]:
vle = tables["vle"].merge(
    tables["vle_meta"][["id_site", "activity_type"]].drop_duplicates("id_site"),
    on="id_site",
    how="left",
)
activity = vle.groupby("activity_type").agg(
    clicks=("sum_click", "sum"),
    records=("sum_click", "size"),
    learners=("id_student", "nunique"),
).sort_values("clicks", ascending=False)
activity.head(15)

In [ ]:
collab_types = ["forumng", "oucollaborate", "ouwiki"]
activity.loc[[t for t in collab_types if t in activity.index]]

Notes:

- `forumng`, `oucollaborate`, and `ouwiki` are treated as explicit collaboration signals downstream.
- `final_result` is intentionally not used for clustering; it is retained only for post-hoc evaluation.